# Part 1: Machine Learning Approach

This notebook represents the first part of the challenge, where we establish a baseline using classical Machine Learning methods for white blood cell classification.

Please make sure that the versions of your installed packages match the ones listed below:

| Library             | Version |
|---------------------|---------|
| NumPy               | 2.4.3   |
| Pandas              | 3.0.1   |
| Matplotlib          | 3.10.8  |
| OpenCV (cv2)        | 4.13.0  |
| SciPy               | 1.17.1  |
| Scikit-Image         | 0.26.0  |
| Scikit-Learn         | 1.8.0   |
| Imbalanced-Learn     | 0.14.1  |

<pre>
data/
├── raw/
│   ├── features_extraites_train.csv
│   ├── features_v2_test.csv
│   └── features_v2_train.csv
├── test/
└── train/
</pre>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import cv2 
from scipy import ndimage
from skimage.filters import threshold_yen, threshold_isodata
from skimage import measure
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
import warnings
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, classification_report, f1_score
os.environ["OPENCV_LOG_LEVEL"] = "SILENT"
warnings.filterwarnings("ignore")
from joblib import Parallel, delayed
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Class Proportion Visualization

In this section, we visualize the dataset's class distribution to help us identify any class imbalances that might affect model performance.

In [ ]:
Working_directory="./data/raw/" 
df = pd.read_csv(Working_directory + 'train_metadata.csv')
display(df.head())

class_counts = df['label'].value_counts()
plt.figure(figsize=(12, 6))
plt.bar(class_counts.index, class_counts.values)
plt.xlabel('Class')
plt.ylabel('Count')
plt.title('Number of samples per class')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
ax = plt.gca()
for bar in ax.patches:
    height = bar.get_height()
    ax.annotate(
        f'{int(height)}',
        (bar.get_x() + bar.get_width() / 2, height),
        ha='center',
        va='bottom',
        xytext=(0, 3),
        textcoords='offset points'
    )

plt.show()


# Segmentation

Here, we apply image processing techniques to segment the white blood cells and their nuclei from the background and red blood cells.

In [ ]:
from scipy import ndimage

def segmenter_cellule_double(image_bgr):
    h, w = image_bgr.shape[:2]
    centre = np.array([w / 2, h / 2])

    image_lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2Lab)
    canal_L = image_lab[:, :, 0]
    canal_b = image_lab[:, :, 2] # Le canal b (bleu-jaune)

    # Les globules blancs sont colorés en violet/bleu (valeurs faibles dans b)
    # Les globules rouges sont plutôt jaunâtres/roses (valeurs élevées dans b)
    # On inverse donc le canal b pour faire ressortir uniquement le globule blanc
    canal_b_inv = 255 - canal_b
    _, masque_brut = cv2.threshold(canal_b_inv, 0, 255,
                                   cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Un lissage pour nettoyer les bords
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    masque_brut = cv2.morphologyEx(masque_brut, cv2.MORPH_CLOSE,
                                   kernel_close, iterations=2)
    masque_brut = ndimage.binary_fill_holes(masque_brut).astype(np.uint8) * 255

    contours, _ = cv2.findContours(masque_brut, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, None

    contours = [c for c in contours if cv2.contourArea(c) > 500]
    if not contours:
        return None, None

    # Score combiné : on favorise les contours grands et proches du centre
    aires = np.array([cv2.contourArea(c) for c in contours], dtype=float)
    
    dists = []
    for c in contours:
        M = cv2.moments(c)
        if M["m00"] == 0:
            dists.append(float('inf'))
        else:
            cx, cy = M["m10"] / M["m00"], M["m01"] / M["m00"]
            dists.append(np.linalg.norm(np.array([cx, cy]) - centre))
    dists = np.array(dists, dtype=float)

    # Normaliser : aire → plus grand = meilleur (max=1), dist → plus petit = meilleur (min=1)
    aires_norm = aires / aires.max() if aires.max() > 0 else aires
    dists_norm = dists / dists.max() if dists.max() > 0 else dists

    # Score = grande aire + faible distance
    scores = 0.5 * aires_norm + 0.5 * (1 - dists_norm)
    contour_cible = contours[np.argmax(scores)]

    masque_cellule_final = np.zeros((h, w), dtype=np.uint8)
    cv2.drawContours(masque_cellule_final, [contour_cible], -1, 255,
                     thickness=cv2.FILLED)

    # --- MASQUE NOYAU : double Otsu sur pixels internes ---
    pixels_cellule = canal_L[masque_cellule_final > 0]
    seuil_noyau, _ = cv2.threshold(pixels_cellule, 0, 255,
                                    cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    masque_noyau = (canal_L < seuil_noyau).astype(np.uint8) * 255
    masque_noyau = cv2.bitwise_and(masque_noyau, masque_cellule_final)
    kernel_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    masque_noyau = cv2.morphologyEx(masque_noyau, cv2.MORPH_CLOSE,
                                    kernel_small, iterations=2)
    masque_noyau = ndimage.binary_fill_holes(masque_noyau).astype(np.uint8) * 255

    return masque_cellule_final, masque_noyau


# Visualisation du résultat de la segmentation
image_bgr = cv2.imread(Working_directory + 'train/train_00001.png')
masque_cellule, masque_noyau = segmenter_cellule_double(image_bgr)

image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
overlay = image_rgb.copy()
overlay[masque_cellule > 0] = (overlay[masque_cellule > 0] * 0.6 +
                                np.array([0, 100, 255]) * 0.4).astype(np.uint8)
overlay[masque_noyau > 0]   = (overlay[masque_noyau > 0]   * 0.6 +
                                np.array([255, 50, 0])  * 0.4).astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(image_rgb);                   axes[0].set_title("Image originale");   axes[0].axis('off')
axes[1].imshow(masque_cellule, cmap='gray'); axes[1].set_title("Masque cellule");    axes[1].axis('off')
axes[2].imshow(masque_noyau,   cmap='gray'); axes[2].set_title("Masque noyau");      axes[2].axis('off')
axes[3].imshow(overlay);                     axes[3].set_title("Superposition\n(bleu=cellule, rouge=noyau)"); axes[3].axis('off')
plt.tight_layout()
plt.show()

# Feature Extraction

We define functions to extract a comprehensive set of morphological, color, and texture features (such as GLCM and LBP) from the segmented regions.

In [ ]:
def extraire_features_morpho(masque_cellule, masque_noyau):
    """
    Features morphologiques sur la cellule entière + le noyau séparément.
    """
    def props_depuis_masque(masque):
        masque_label = measure.label(masque > 0)
        props = measure.regionprops(masque_label)
        return props[0] if props else None

    cellule = props_depuis_masque(masque_cellule)
    noyau   = props_depuis_masque(masque_noyau)

    if cellule is None:
        return None

    aire_cellule = cellule.area
    perimetre    = cellule.perimeter
    aire_noyau   = noyau.area if noyau else 0
    aire_cyto    = aire_cellule - aire_noyau

    circularite_cellule = (4 * np.pi * aire_cellule) / (perimetre ** 2) if perimetre > 0 else 0

    min_row, min_col, max_row, max_col = cellule.bbox
    hauteur_bbox = max_row - min_row
    largeur_bbox = max_col - min_col
    rapport_aspect = largeur_bbox / hauteur_bbox if hauteur_bbox > 0 else 1

    # Hu moments sur la cellule
    moments_hu = cv2.HuMoments(cv2.moments(masque_cellule)).flatten()
    hu_log = [-np.sign(h) * np.log10(abs(h)) if h != 0 else 0 for h in moments_hu]

    features = {
        # Cellule entière
        "Surface_cellule":      aire_cellule,
        "Perimetre_cellule":    perimetre,
        "Excentricite_cellule": cellule.eccentricity,
        "Solidite_cellule":     cellule.solidity,
        "Circularite_cellule":  circularite_cellule,
        "Axe_majeur_cellule":   cellule.major_axis_length,
        "Rapport_aspect":       rapport_aspect,
        # Noyau
        "Surface_noyau":        aire_noyau,
        "Circularite_noyau":    (4 * np.pi * aire_noyau) / (noyau.perimeter ** 2)
                                 if noyau and noyau.perimeter > 0 else 0,
        "Excentricite_noyau":   noyau.eccentricity if noyau else 0,
        "Solidite_noyau":       noyau.solidity      if noyau else 0,
        # Ratios noyau / cellule 
        "Ratio_noyau_cellule":  aire_noyau / aire_cellule if aire_cellule > 0 else 0,
        "Surface_cytoplasme":   aire_cyto,
        "Ratio_cyto_cellule":   aire_cyto  / aire_cellule if aire_cellule > 0 else 0,
    }

    for i in range(7):
        features[f"Hu_{i+1}"] = hu_log[i]

    return features

In [ ]:
def extraire_features_couleur(image_bgr, masque_cellule, masque_noyau):
    """
    Stats couleur RGB+HSV sur : cellule entière, noyau seul, cytoplasme seul.
    """
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image_hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)

    # Masque cytoplasme = cellule - noyau
    masque_cyto = cv2.bitwise_and(masque_cellule,
                                   cv2.bitwise_not(masque_noyau))

    features = {}
    regions = {
        "cellule": masque_cellule > 0,
        "noyau":   masque_noyau   > 0,
        "cyto":    masque_cyto    > 0,
    }

    for region_nom, mask_bool in regions.items():
        if not mask_bool.any():
            for canal in ["R", "G", "B", "H", "S", "V"]:
                features[f"Moyenne_{canal}_{region_nom}"] = 0
                features[f"Std_{canal}_{region_nom}"]     = 0
            continue

        for i, canal in enumerate(["R", "G", "B"]):
            pixels = image_rgb[:, :, i][mask_bool].astype(float)
            features[f"Moyenne_{canal}_{region_nom}"] = np.mean(pixels)
            features[f"Std_{canal}_{region_nom}"]     = np.std(pixels)

        for i, canal in enumerate(["H", "S", "V"]):
            pixels = image_hsv[:, :, i][mask_bool].astype(float)
            features[f"Moyenne_{canal}_{region_nom}"] = np.mean(pixels)
            features[f"Std_{canal}_{region_nom}"]     = np.std(pixels)

    # Ratios sur la cellule entière
    mr = features["Moyenne_R_cellule"]
    mg = features["Moyenne_G_cellule"]
    mb = features["Moyenne_B_cellule"]
    features["Ratio_RG"] = mr / mg if mg > 0 else 0
    features["Ratio_RB"] = mr / mb if mb > 0 else 0
    features["Ratio_GB"] = mg / mb if mb > 0 else 0

    return features


def extraire_features_glcm(image_gray, masque):
    coords = np.where(masque > 0)
    if len(coords[0]) == 0:
        return None
    min_r, max_r = coords[0].min(), coords[0].max()
    min_c, max_c = coords[1].min(), coords[1].max()
    cellule_crop = image_gray[min_r:max_r+1, min_c:max_c+1]
    if cellule_crop.shape[0] < 2 or cellule_crop.shape[1] < 2:
        return None
    angles    = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    angle_noms = ["0", "45", "90", "135"]
    proprietes = ["contrast", "homogeneity", "energy", "correlation", "dissimilarity"]
    glcm = graycomatrix(cellule_crop, distances=[1], angles=angles,
                        levels=256, symmetric=True, normed=True)
    features = {}
    for prop in proprietes:
        vals = graycoprops(glcm, prop)[0]
        for a, angle_nom in enumerate(angle_noms):
            features[f"GLCM_{prop}_{angle_nom}"] = vals[a]
    return features


def extraire_features_lbp(image_gray, masque):
    P, R = 8, 1
    lbp = local_binary_pattern(image_gray, P, R, method="uniform")
    lbp_cellule = lbp[masque > 0]
    n_bins = P + 2
    hist, _ = np.histogram(lbp_cellule, bins=n_bins, range=(0, n_bins), density=True)
    return {f"LBP_bin_{i}": hist[i] for i in range(n_bins)}


def extraire_toutes_features(chemin_image):
    os.environ["OPENCV_LOG_LEVEL"] = "OFF"
    try:
        image_bgr = cv2.imread(chemin_image)
        if image_bgr is None:
            return None

        masque_cellule, masque_noyau = segmenter_cellule_double(image_bgr)
        if masque_cellule is None:
            return None

        image_gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

        f_morpho  = extraire_features_morpho(masque_cellule, masque_noyau)
        f_couleur = extraire_features_couleur(image_bgr, masque_cellule, masque_noyau)
        f_glcm    = extraire_features_glcm(image_gray, masque_cellule)
        f_lbp     = extraire_features_lbp(image_gray, masque_cellule)

        if any(f is None for f in [f_morpho, f_couleur, f_glcm, f_lbp]):
            return None

        features = {}
        features.update(f_morpho)
        features.update(f_couleur)
        features.update(f_glcm)
        features.update(f_lbp)
        return features

    except Exception:
        return None

import os
import pandas as pd
from joblib import Parallel, delayed

def traiter_une_image(row, dossier_images, i):
    """Fonction helper pour traiter une seule image"""
    if (i + 1) % 1000 == 0:
        print(f"{i + 1} images traitées...")
        
    chemin = os.path.join(dossier_images, row["ID"])
    features = extraire_toutes_features(chemin)
    
    if features is not None:
        features["ID"] = row["ID"]
        if "label" in row:
            features["label"] = row["label"]
        return features
    return None

def extraire_dataset_parallel(dossier_images, metadata_csv, output_csv, n_jobs=-1):
    """
    Extrait les features en parallèle.
    """
    df_meta = pd.read_csv(metadata_csv)
    print(f"Extraction en parallèle pour {len(df_meta)} images")
    
    # Exécution en parallèle 
    resultats_bruts = Parallel(n_jobs=n_jobs)(
        delayed(traiter_une_image)(row, dossier_images, i) for i, row in df_meta.iterrows()
    )
    
    # Nettoyage des résultats 
    resultats = [res for res in resultats_bruts if res is not None]
    echecs = len(df_meta) - len(resultats)
    
    # Création du dataframe et sauvegarde
    df_result = pd.DataFrame(resultats)
    df_result.to_csv(output_csv, index=False)

    print(f"\nTerminé ! {len(resultats)} images traitées avec succès, {echecs} échecs.")
    print(f"Nombre de features : {len(df_result.columns) - 2}")
    print(f"Sauvegardé dans : {output_csv}")

    return df_result

# Full Dataset Processing

This section runs the feature extraction pipeline in parallel across the entire training and testing datasets to efficiently build our tabular data.

In [ ]:
WORKING_DIR = "./data/raw/"
NB_COEURS = 16

df_train = extraire_dataset_parallel(
        dossier_images=os.path.join(WORKING_DIR, "train"),
        metadata_csv=os.path.join(WORKING_DIR, "train_metadata.csv"),
        output_csv=os.path.join(WORKING_DIR, "features_v2_train.csv"),
        n_jobs=NB_COEURS
    )

df_test = extraire_dataset_parallel(
        dossier_images=os.path.join(WORKING_DIR, "test"),
        metadata_csv=os.path.join(WORKING_DIR, "test_metadata.csv"),
        output_csv=os.path.join(WORKING_DIR, "features_v2_test.csv"),
        n_jobs=NB_COEURS
    )

print(f"Shape: {df_train.shape}")
print(f"\nColonnes:\n{list(df_train.columns)}")
print(f"\nDistribution des classes:")
if "label" in df_train.columns:
    print(df_train["label"].value_counts())

# Cross-Validation on Multiple Models

We compare several machine learning classification algorithms using cross-validation to identify the most promising model for our extracted features.

In [ ]:
df = pd.read_csv(Working_directory + 'features_v2_train.csv')
X, y = df.drop(['label', 'ID'], axis=1), df['label']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

models = {
    "Régression Logistique": LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1),
    "KNN": KNeighborsClassifier(n_jobs=-1), 
    "SVM (RBF)": SVC(class_weight='balanced'),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

# Validation croisée
results = []
for name, model in models.items():
    cv_scores = cross_validate(model, X_scaled, y, cv=5, scoring=('accuracy', 'f1_macro'), n_jobs=-1)
    
    results.append({
        "Modèle": name,
        "Accuracy Moyenne": cv_scores['test_accuracy'].mean(),
        "F1-Score (Macro)": cv_scores['test_f1_macro'].mean()
    })

df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="F1-Score (Macro)", ascending=False).reset_index(drop=True)

print(df_results.to_string())


In [ ]:
classes = sorted(y.unique())

fig, axes = plt.subplots(2, 3, figsize=(22, 14))
axes = axes.flatten()
 
for idx, (name, model) in enumerate(models.items()):
    print(f"Calcul des prédictions pour {name}")
    y_pred = cross_val_predict(model, X_scaled, y, cv=5, n_jobs=-1)
    
    cm = confusion_matrix(y, y_pred, labels=classes)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)  
    
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=classes)
    disp.plot(ax=axes[idx], cmap="Blues", values_format=".2f", colorbar=False)
    axes[idx].set_title(name, fontsize=13, fontweight="bold")
    axes[idx].set_xlabel("Prédit")
    axes[idx].set_ylabel("Réel")
    axes[idx].set_xticklabels(classes, rotation=45, ha="right", fontsize=8)
    axes[idx].set_yticklabels(classes, fontsize=8)
 
plt.suptitle("Matrices de confusion normalisées pour chaque model", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("Meilleur modèle")
best_model = list(models.values())[list(models.keys()).index("Gradient Boosting")]
y_pred_best = cross_val_predict(best_model, X_scaled, y, cv=5, n_jobs=-1)
print(classification_report(y, y_pred_best, digits=3))

In [ ]:
df_train = pd.read_csv(Working_directory + 'features_v2_train.csv')
df_test = pd.read_csv(Working_directory + 'features_v2_test.csv')

feature_cols = [c for c in df_train.columns if c not in ['ID', 'label']]

X_train = df_train[feature_cols]
y_train = df_train['label']
X_test = df_test[feature_cols]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) 

model = GradientBoostingClassifier(random_state=42)
model.fit(X_train_scaled, y_train)

# on prédie le test 
y_pred = model.predict(X_test_scaled)
# on crée le fichier de submission
submission = pd.DataFrame({
    'ID': df_test['ID'],
    'label': y_pred
})

submission.to_csv(Working_directory + 'submission.csv', index=False)
print(f"Submission créée : {len(submission)} prédictions")
print(submission.head(10))
print(f"\nDistribution des prédictions :")
print(submission['label'].value_counts())


# Optimization with SMOTE

To tackle the class imbalance observed earlier, we introduce SMOTE within a pipeline and optimize our best-performing model's hyperparameters using Grid Search.

In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV

df = pd.read_csv(Working_directory + 'features_v2_train.csv')
X = df.drop(['label', 'ID'], axis=1)
y = df['label']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#  Validation croisée 
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
smote = SMOTE(random_state=42, k_neighbors=3)

print("GRID SEARCH: Gradient Boosting + SMOTE")

pipe_gb = ImbPipeline([
    ("smote", smote),
    ("model", GradientBoostingClassifier(random_state=42))
])

param_grid_gb = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [3, 5],
    "model__learning_rate": [0.05, 0.1],
}

grid_gb = GridSearchCV(
    pipe_gb,
    param_grid_gb,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,     
    verbose=1,
    refit=True
)

grid_gb.fit(X_scaled, y)

print(f"\nMeilleurs paramètres : {grid_gb.best_params_}")
print(f"Meilleur F1 macro : {grid_gb.best_score_:.4f}")

# Évaluation du modèle avec les meilleurs paramètres
best_pipe = grid_gb.best_estimator_
best_name = "Gradient Boosting"

y_pred = cross_val_predict(best_pipe, X_scaled, y, cv=cv, n_jobs=-1)

classes = sorted(y.unique())
cm = confusion_matrix(y, y_pred, labels=classes)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)

fig, ax = plt.subplots(figsize=(12, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=classes)
disp.plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=True)
ax.set_title(f"Matrice de confusion: {best_name} + SMOTE", fontsize=14, fontweight="bold")
ax.set_xlabel("Prédit", fontsize=12)
ax.set_ylabel("Réel", fontsize=12)
ax.set_xticklabels(classes, rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(f"\n{best_name} + SMOTE")
print(classification_report(y, y_pred, digits=3))

# Prediction

Finally, we train our optimized model on the complete training dataset, generate predictions for the test dataset, and save the results for submission.

In [ ]:
df_train = pd.read_csv(Working_directory + 'features_v2_train.csv')
df_test = pd.read_csv(Working_directory + 'features_v2_test.csv')

feature_cols = [c for c in df_train.columns if c not in ['ID', 'label']]

X_train = df_train[feature_cols]
y_train = df_train['label']
X_test = df_test[feature_cols]
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

best_pipe.fit(X_train_scaled, y_train)

# Prédire sur le test
y_pred_test = best_pipe.predict(X_test_scaled)

# Créer le fichier de submission
submission = pd.DataFrame({
    'ID': df_test['ID'],
    'label': y_pred_test
})

submission.to_csv(Working_directory + 'submission_optimized.csv', index=False)
print(f"Submission créée : {len(submission)} prédictions")
print(submission.head(10))
print(f"Distribution des prédictions :")
print(submission['label'].value_counts())